# Доверительные интервалы для доли (конверсии)

По разделам «Доверительные интервалы» из [лекции Stat power.ipynb Авито](https://github.com/avito-tech/applied_statistics/blob/main/Stat%20power.ipynb).

**Два способа:** перебор допустимых значений вероятности по биномиальному критерию и быстрый интервал Уилсона. Дополнительно — односторонняя нижняя граница.

**Как пользоваться:** поменяй числа в следующей ячейке, затем выполни **Run All**.


In [ ]:
from scipy.stats import binom
from statsmodels.stats.proportion import proportion_confint
import numpy as numpy
from matplotlib import pyplot


## 1. ВВОД ДАННЫХ


In [ ]:
N = 30              # всего наблюдений
SUCCESS_CNT = 19     # из них успешных
ALPHA = 0.05         # уровень значимости (0.05 = доверие 95%)
GRID_STEP = 0.001    # шаг перебора возможных вероятностей
SHOW_PLOTS = True    # False — только числа


## 2. Доверительный интервал перебором

Перебираем значения μ₀, для которых H₀ не отвергается. Ниже — функции исходного конспекта; исправлены только жёстко заданные `n=30` и `alpha=0.05`, чтобы можно было менять параметры.


In [ ]:
# Функция из прошлой лекции
def two_sided_criterion_nonsym(n, mu, alpha):
    '''Строит двусторонний критерий для несимметричной задачи с доставкой
    
    Параметры:
        n: количество доставок в эксперименте
        mu: вероятность успеха в нулевой гипотезе
        alpha: уровень значимости критерия
        
    Возвращает:
        C1, C2 для критерия S = {Q <= C1 или Q >= C2}
    '''
    binom_h0 = binom(n=n, p=mu)
    
    # аналогично одностороннему критерию
    c2 = binom_h0.ppf(1 - alpha/2) + 1
        
    # по выкладкам выше
    c1 = binom_h0.ppf(alpha/2) - 1
    
    return c1, c2

In [ ]:
def my_binomial_confint(n, alpha, q, grid_step=0.001):
    '''Вычисляет доверительный интервал для выборки из распределения Бернулли
    
        n (int) - количество элементов в выборке
        alpha (float) - уровень значимости критерия
        q (float) - реализация статистики критерия
    '''
    mu_grid = numpy.linspace(0, 1, int(round(1 / grid_step)) + 1)
    mu_no_rejection = []

    for mu_h0 in mu_grid:
        c1, c2 = two_sided_criterion_nonsym(n, mu_h0, alpha=alpha)
        if q > c1 and q < c2:
            mu_no_rejection.append(mu_h0)

    if not mu_no_rejection:
        raise ValueError("Интервал не найден: уменьшите grid_step")
    return min(mu_no_rejection), max(mu_no_rejection)

In [ ]:
left_grid, right_grid = my_binomial_confint(
    n=N, alpha=ALPHA, q=SUCCESS_CNT, grid_step=GRID_STEP
)
print(f"Интервал перебором: {left_grid:.3f} — {right_grid:.3f}")


## 3. Доверительный интервал Уилсона

В лекции используется функция `proportion_confint(..., method='wilson')`.


In [ ]:
left_wilson, right_wilson = proportion_confint(
    count=SUCCESS_CNT, # количество успехов
    nobs=N,             # размер выборки
    alpha=ALPHA,
    method='wilson'
)
print(f"Интервал Уилсона: {left_wilson:.3f} — {right_wilson:.3f}")


## 4. Односторонняя нижняя доверительная граница


In [ ]:
def make_binom_criterion(n, mu=0.5, alpha=0.05):
    '''Строит критерий для задачи с доставкой
    
    Параметры:
        n: количество доставок в эксперименте
        mu: вероятность успеха в нулевой гипотезе
        alpha: уровень значимости критерия
        
    Возвращает:
        C для критерия S = {Q >= C}
    '''
    binom_h0 = binom(n=n, p=mu)
    q = binom_h0.ppf(1 - alpha)
    return q + 1

In [ ]:
mu_grid = numpy.linspace(0, 1, int(round(1 / GRID_STEP)) + 1)
mu_no_rejection = []
for mu_h0 in mu_grid:
    crit_val = make_binom_criterion(n=N, mu=mu_h0, alpha=ALPHA)
    if SUCCESS_CNT < crit_val:
        mu_no_rejection.append(mu_h0)
if not mu_no_rejection:
    raise ValueError("Нижняя граница не найдена: уменьшите GRID_STEP")
lower_grid = min(mu_no_rejection)

# Двусторонний интервал Уилсона с alpha*2 даёт одностороннюю нижнюю границу.
lower_wilson, _ = proportion_confint(
    count=SUCCESS_CNT, nobs=N, alpha=2*ALPHA, method="wilson"
)
print(f"Нижняя граница (перебор): {lower_grid:.3f}")
print(f"Нижняя граница (Уилсон):  {lower_wilson:.3f}")


## 5. График: сравнение интервалов


In [ ]:
if SHOW_PLOTS:
    observed_share = SUCCESS_CNT / N
    pyplot.figure(figsize=(10, 4.5))
    pyplot.hlines(2, left_grid, right_grid, linewidth=6, label="Перебор")
    pyplot.hlines(1, left_wilson, right_wilson, linewidth=6, label="Уилсон")
    pyplot.scatter([left_grid, right_grid], [2, 2], s=35)
    pyplot.scatter([left_wilson, right_wilson], [1, 1], s=35)
    pyplot.axvline(observed_share, ls="--", label=f"Доля в выборке: {observed_share:.1%}")
    pyplot.yticks([1, 2], ["Уилсон", "Перебор"])
    pyplot.xlim(0, 1)
    pyplot.xlabel("Вероятность успеха")
    pyplot.title(f"Доверительные интервалы ({1-ALPHA:.0%})")
    pyplot.legend()
    pyplot.show()
